Name: Bryce Kugler
## Data Science Applications Project - Aggregate Movie Rating Website Comparison
The purpose of this project is to compare the rating values for movies on various aggregate movie rating websites, including Letterboxd, IMDb, Rotten Tomatoes and Metacritic.

What this entails is creating a database consisting of a large list of movies, basic information about them (release year and genre), and their aggregate rating scores for each website.

Statistics summaries and visualizations in this project are from a database consisting of 9,000+ movies with no missing values. Statistics/visualizations include summary statistics (mean, standard deviation, percentiles) and visualizations of said summary statistics.

Summary statistics and visualizations are also included for each individual genre, as it is a major factor in comparing aggregate rating websites. A similar thing is done for release years, though only mean values are included and not all summary statistics, as it is not a major focus like genre.


In [4]:
!pip install kagglehub[pandas-datasets]

In [57]:
# Importing necessary functions
import kagglehub # For loading databases
from kagglehub import KaggleDatasetAdapter
import pandas as pd # For creating and editing dataframes
import numpy as np # For various mathematical functions
import plotly.express as px # For data visualization
import plotly.graph_objects as go
from ast import literal_eval # Singular function for converting list-like string to list

# Included in the kagglehub.dataset_load function is four arguments:
# 1. KaggleDatasetAdapter.PANDAS chooses which type of database for the data to be loaded into, in this case Pandas
# 2. A file path for the dataset to be loaded. Notice how each instance of the argument is also in the link for the dataset on the Kaggle website.
# 3. Which database to be selected from based on the link provided in the second argument
# 4. pandas_kwargs is a function for selecting individual columns to be selected, among other things.
lbxd_df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "gsimonx37/letterboxd", "movies.csv", pandas_kwargs={"usecols": ["id", "name", "date", "rating"]})
# Link: https://www.kaggle.com/datasets/gsimonx37/letterboxd?select=movies.csv
imdb_df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "utsh0dey/25k-movie-dataset", "25k IMDb movie Dataset.csv", pandas_kwargs={"usecols": ["movie title", "Rating", "User Rating", "Generes"]})
# Link: https://www.kaggle.com/datasets/utsh0dey/25k-movie-dataset?select=25k+IMDb+movie+Dataset.csv
rt_df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "andrezaza/clapper-massive-rotten-tomatoes-movies-and-reviews", "rotten_tomatoes_movies.csv", pandas_kwargs={"usecols": ["title", "audienceScore", "tomatoMeter"]})
# Link: https://www.kaggle.com/datasets/andrezaza/clapper-massive-rotten-tomatoes-movies-and-reviews?select=rotten_tomatoes_movies.csv
mc_df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "mohamedasak/metacritic-movies-dataset", "metacritic_movies.csv", pandas_kwargs={"usecols": ["title", "metascore"]})
# Link: https://www.kaggle.com/datasets/mohamedasak/metacritic-movies-dataset?select=metacritic_movies.csv

Using Colab cache for faster access to the '25k-movie-dataset' dataset.
Using Colab cache for faster access to the 'metacritic-movies-dataset' dataset.


In [7]:
# Displaying head & tail of each database
display(lbxd_df)
display(imdb_df)
display(rt_df)
display(mc_df)

,id,name,date,rating
0,1000001,Barbie,2023.0,3.86
1,1000002,Parasite,2019.0,4.56
2,1000003,Everything Everywhere All at Once,2022.0,4.30
3,1000004,Fight Club,1999.0,4.27
4,1000005,La La Land,2016.0,4.09
...,...,...,...,...
941592,1941593,神笛,NaN,NaN
941593,1941594,蟲極道蜜団子抗争編 壱ノ巻,NaN,NaN
941594,1941595,蟲極道蜜団子抗争編 弐ノ巻,NaN,NaN
941595,1941596,重生,NaN,NaN


,movie title,Rating,User Rating,Generes
0,Top Gun: Maverick,8.6,187K,"['Action', 'Drama']"
1,Jurassic World Dominion,6,56K,"['Action', 'Adventure', 'Sci-Fi']"
2,Top Gun,6.9,380K,"['Action', 'Drama']"
3,Lightyear,5.2,32K,"['Animation', 'Action', 'Adventure']"
4,Spiderhead,5.4,23K,"['Action', 'Crime', 'Drama']"
...,...,...,...,...
24397,Delicatessen,7.6,85K,"['Comedy', 'Crime']"
24398,Bitch Ass,5.5,52,"['Crime', 'Horror']"
24399,Bullwhip,5.1,398,"['Crime', 'Romance', 'Western']"
24400,The Freshman,6.4,20K,"['Comedy', 'Crime']"


,title,audienceScore,tomatoMeter
0,Space Zombie Bingo!,50.0,NaN
1,The Green Grass,NaN,NaN
2,"Love, Lies",43.0,NaN
3,Sore Losers,60.0,NaN
4,Dinosaur Island,70.0,NaN
...,...,...,...
143253,Nadia: The Secret of Blue Water: The Motion Pi...,14.0,NaN
143254,Everyone I Knew and Loved,NaN,NaN
143255,The Human Body,71.0,89.0
143256,Flying Fists,NaN,NaN


,title,metascore
0,Dekalog (1988),100.0
1,Citizen Kane,100.0
2,Boyhood,100.0
3,The Leopard (re-release),100.0
4,The Godfather,100.0
...,...,...
16934,The Burning Wall,NaN
16935,Boogie Woogie,NaN
16936,Saint Misbehavin': The Wavy Gravy Movie,NaN
16937,Afghan Stories,NaN


In [8]:
# Database Cleanup Part 1: User Ratings
# "User Ratings" column from the IMDb database shows how many users have rated a movie, included to filter out very obscure movies (5000 or less reviews)
# User ratings are formated 100k, 1.2m, etc., so the below function converts them to integers
def convert_k_and_m_to_int(x):
    x = str(x).lower()
    if 'k' in x:
        return int(float(x.replace('k', '')) * 1000)
    if 'm' in x:
        return int(float(x.replace('m', '')) * 1000000)
    return int(x)

# Above function is applied, filter is made, and User Rating column is removed since I have no need for it
imdb_df['User Rating'] = imdb_df['User Rating'].apply(convert_k_and_m_to_int)
imdb_df = imdb_df[imdb_df['User Rating'] >= 5000]
imdb_df = imdb_df.drop(columns=['User Rating'])
display(imdb_df)

,movie title,Rating,Generes
0,Top Gun: Maverick,8.6,"['Action', 'Drama']"
1,Jurassic World Dominion,6,"['Action', 'Adventure', 'Sci-Fi']"
2,Top Gun,6.9,"['Action', 'Drama']"
3,Lightyear,5.2,"['Animation', 'Action', 'Adventure']"
4,Spiderhead,5.4,"['Action', 'Crime', 'Drama']"
...,...,...,...
24394,The Long Riders,6.9,"['Biography', 'Crime', 'Western']"
24396,Police Academy 4: Citizens on Patrol,4.9,"['Comedy', 'Crime']"
24397,Delicatessen,7.6,"['Comedy', 'Crime']"
24400,The Freshman,6.4,"['Comedy', 'Crime']"


In [9]:
# Database Cleanup Part 1.5: Name Fixing
# Renaming columns on each database accordingly
lbxd_df = lbxd_df.rename(columns={'name': 'movie_title', 'rating': 'lbxd_rating'})
imdb_df = imdb_df.rename(columns={'Rating': 'imdb_rating', 'Generes': 'genre'})
rt_df = rt_df.rename(columns={'audienceScore': 'rt_rating_audience', 'tomatoMeter': 'rt_rating_critic'})
mc_df = mc_df.rename(columns={'metascore': 'mc_rating_critic'})

# Database Cleanup Part 2: Merging
# Starting off by merging the Letterboxd and IMDb tables by movie title, then removing the unnessecary title column, then doing the same with the Rotten Tomatoes and Metacritic database
# Letterboxd and IMDb are merged first, as the Letterboxd database has the most data, while the IMDb dataset includes information on genre and movie popularity, which has already been utilized to filter out obscure movies
combined_df = pd.merge(lbxd_df, imdb_df, left_on='movie_title', right_on='movie title', how='inner')
combined_df = combined_df.drop(columns=['movie title'])
combined_df = pd.merge(combined_df, rt_df, left_on='movie_title', right_on='title', how='inner')
combined_df = combined_df.drop(columns=['title'])
combined_df = pd.merge(combined_df, mc_df, left_on='movie_title', right_on='title', how='inner')
combined_df = combined_df.drop(columns=['title'])

# Database Cleanup part 3: Misc. Cleanup
combined_df = combined_df.dropna(subset=combined_df.columns.difference(['date'])).drop_duplicates(subset=['id']).reset_index(drop='True') # Removing null values and removing duplicate movie columns based on the Letterboxd database's ID column (date column is exempt from dropna)
combined_df = combined_df.drop(columns=['id']) # Removing ID column since I have no need for it
genre_col = combined_df.pop('genre') # Moving genre column to proper place
combined_df.insert(2, 'genre', genre_col)
combined_df = combined_df[combined_df['imdb_rating'] != 'no-rating'] # Removing rows with "no-rating" value that appeared occasionally in IMDb database
combined_df['lbxd_rating'] = combined_df['lbxd_rating'] * 20 # Adjusting rating values since each website uses a different metric (0-5/5 for Letterboxd and 0-10/10 for IMDb)
combined_df['imdb_rating'] = combined_df['imdb_rating'].astype(float) * 10
combined_df['genre'] = combined_df['genre'].apply(literal_eval) # Converts the string that looks like a list in the genre column into an actual list
movie_rating_data = combined_df.sort_values(by='movie_title').reset_index(drop='True') # Ordering database by movie title and giving it final variable name

display(movie_rating_data) # After all is said and done, the database has 9,000+ movies, including their title, date, genres, and normalized ratings for all four websites, without any null values.

,movie_title,date,genre,lbxd_rating,imdb_rating,rt_rating_audience,rt_rating_critic,mc_rating_critic
0,10,1979.0,"[Comedy, Romance]",63.0,61.0,53.0,68.0,68.0
1,10 Cloverfield Lane,2016.0,"[Drama, Horror, Mystery]",71.4,72.0,79.0,90.0,76.0
2,10 Things I Hate About You,1999.0,"[Comedy, Drama, Romance]",79.8,73.0,69.0,71.0,70.0
3,10 Years,2011.0,"[Comedy, Drama, Romance]",57.4,61.0,40.0,60.0,61.0
4,101 Dalmatians,1996.0,"[Adventure, Comedy, Crime]",56.4,57.0,40.0,41.0,83.0
...,...,...,...,...,...,...,...,...
9810,Zootopia,2016.0,"[Animation, Adventure, Comedy]",74.0,80.0,92.0,98.0,78.0
9811,Zulu,2013.0,"[Drama, History, War]",65.0,77.0,91.0,96.0,77.0
9812,Zulu,1964.0,"[Drama, History, War]",75.0,77.0,91.0,96.0,77.0
9813,eXistenZ,1999.0,"[Horror, Mystery, Sci-Fi]",71.8,68.0,69.0,74.0,70.0


In [10]:
# Data Visualizations Part 1: Entire Dataset
print("Summary of dataset")
display(movie_rating_data.describe()) # Provides basic statistical info

print("\nList of genres")
genre_list = movie_rating_data['genre'].explode().unique() # .explode() converts column into a long list (since values in the genre column are lists)
print(genre_list)

# px.bar is used to create an interactive bar graph based on provided data, with a title and labels
# range_y is used to limit the range of values on the y-axis
# update_layout and update_traces are used to make adjustments to the created bar graph
print("\n")
rating_means = px.bar(movie_rating_data.describe().iloc[1, 1:6].sort_values(), title="Average rating value for each website", labels={'index': 'websites', 'value': 'mean'}, range_y=[50,70])
rating_means.update_layout(showlegend=False, width=800)
display(rating_means)

print("\n")
rating_std = px.bar(movie_rating_data.describe().iloc[2, 1:6].sort_values(), title="Standard deviation of rating values for each website", labels={'index': 'websites', 'value': 'standard deviation'})
rating_std.update_layout(showlegend=False, width=800)
rating_std.update_traces(marker_color='red')
display(rating_std)

Summary of dataset


,date,lbxd_rating,imdb_rating,rt_rating_audience,rt_rating_critic,mc_rating_critic
count,9815.000000,9815.000000,9815.000000,9815.000000,9815.000000,9815.000000
mean,2000.519715,63.449455,64.788487,62.080489,60.644014,58.965155
std,19.980304,10.225966,9.173313,20.067941,27.475587,17.835646
min,1891.000000,21.400000,17.000000,0.000000,0.000000,1.000000
25%,1992.000000,57.000000,59.000000,46.000000,38.000000,46.000000
50%,2006.000000,64.200000,65.000000,64.000000,66.000000,60.000000
75%,2015.000000,70.400000,71.000000,79.000000,85.000000,72.000000
max,2024.000000,92.800000,93.000000,100.000000,100.000000,100.000000



List of genres
['Comedy' 'Romance' 'Drama' 'Horror' 'Mystery' 'Adventure' 'Crime'
 'Family' 'Thriller' 'Action' 'History' 'Biography' 'Fantasy' 'War'
 'Sci-Fi' 'Western' 'Sport' 'Musical' 'Music' 'Animation' 'Film-Noir']




In [11]:
print("\nGenre count and percent ratios")
display(movie_rating_data['genre'].explode().value_counts().to_frame(name='count').assign(percentage=movie_rating_data['genre'].explode().value_counts(normalize=True) * 100)) # .to_frame() and .assign() are used to display count and percentage column next to eachother

# Creating unique value curves for each numeric column, used for displaying how the dataset is distributed
print("\n")
year_uvcurve = px.bar(movie_rating_data['date'].value_counts(), title="Unique Value Curve: Date", labels={"value": "count"})
year_uvcurve.update_layout(showlegend=False)
year_uvcurve.update_traces(marker_color='dimgray')
display(year_uvcurve)
print("\n")

lbxd_uvcurve = px.bar(movie_rating_data['lbxd_rating'].round().value_counts(), title="Unique Value Curve: Letterboxd rating") # .round() is applied only to letterboxd ratings to keep bar width consistent across graphs
lbxd_uvcurve.update_layout(showlegend=False)
lbxd_uvcurve.update_traces(marker_color='orange')
display(lbxd_uvcurve)
print("\n")

imdb_uvcurve = px.bar(movie_rating_data['imdb_rating'].value_counts(), title="Unique Value Curve: IMDb rating", labels={"value": "count"})
imdb_uvcurve.update_layout(showlegend=False)
imdb_uvcurve.update_traces(marker_color='gold')
display(imdb_uvcurve)
print("\n")

rt_aud_uvcurve = px.bar(movie_rating_data['rt_rating_audience'].value_counts(), title="Unique Value Curve: Rotten Tomatoes audience rating", labels={"value": "count"})
rt_aud_uvcurve.update_layout(showlegend=False)
rt_aud_uvcurve.update_traces(marker_color='tomato')
display(rt_aud_uvcurve)
print("\n")

rt_crit_uvcurve = px.bar(movie_rating_data['rt_rating_critic'].value_counts(), title="Unique Value Curve: Rotten Tomatoes critic rating", labels={"value": "count"})
rt_crit_uvcurve.update_layout(showlegend=False)
rt_crit_uvcurve.update_traces(marker_color='darkred')
display(rt_crit_uvcurve)
print("\n")

mc_uvcurve = px.bar(movie_rating_data['mc_rating_critic'].value_counts(), title="Unique Value Curve: Metacritic critic rating", labels={"value": "count"})
mc_uvcurve.update_layout(showlegend=False)
mc_uvcurve.update_traces(marker_color='darkgreen')
display(mc_uvcurve)
print("\n")


Genre count and percent ratios


,count,percentage
genre,,
Drama,5949,23.319352
Comedy,3249,12.735683
Action,2422,9.493944
Crime,2105,8.251343
Thriller,1836,7.196895
Romance,1738,6.812747
Adventure,1636,6.412920
Horror,1149,4.503939
Mystery,1078,4.225628


In [12]:
# Data Visualizations Part 2: By Genre

movie_rating_data_exploded = movie_rating_data.explode('genre') # Exploding movie_rating_data so genre column is separated into strings

# For statistical summaries of each column by genre, I created a table grouped by genres in rows, with the statistical values as columns
genre_summaries_list = [] # Creating a list of summary dataframes for each genre
for genre in genre_list:
  genre_summaries_list.append(movie_rating_data_exploded.loc[movie_rating_data_exploded['genre'] == genre].describe())

genre_summaries = pd.concat(genre_summaries_list, keys=genre_list, axis=1).transpose() # Concatenating list of summaries and switching rows and columns
genre_summaries.drop(columns=["count"], axis=1, inplace=True) # Removing count column since it's unnessecary
print("Statistical summaries for each genre")
display(genre_summaries.iloc[0:42]) # In order to display the entire set, I need three different display function calls
display(genre_summaries.iloc[42:84])
display(genre_summaries.iloc[84:126])


Statistical summaries for each genre


mean        std     min      25%     50%  \
Comedy    date                2001.019391  18.240371  1900.0  1993.00  2005.0   
          lbxd_rating           62.150693  10.147798    22.8    55.60    62.8   
          imdb_rating           63.175131   9.074801    19.0    58.00    64.0   
          rt_rating_audience    60.986457  18.929749     9.0    46.00    62.0   
          rt_rating_critic      56.846414  28.141326     0.0    33.00    60.0   
          mc_rating_critic      55.425054  17.789149     1.0    43.00    56.0   
Romance   date                1998.931530  21.105416  1895.0  1992.00  2005.0   
          lbxd_rating           63.710012   9.368156    24.2    57.80    64.2   
          imdb_rating           64.997123   8.431340    19.0    60.00    66.0   
          rt_rating_audience    62.642693  18.121307    13.0    49.00    65.0   
          rt_rating_critic      59.699079  27.540828     0.0    37.00    64.0   
          mc_rating_critic      58.812428  18.023730     7.0    46.00    59.0   
Drama     date                2000.628845  20.467216  1891.0  1993.00  2007.0   
          lbxd_rating           65.813414   9.199801    25.4    60.00    66.4   
          imdb_rating           66.919987   8.231232    23.0    62.00    67.0   
          rt_rating_audience    65.306270  19.377918     0.0    51.00    69.0   
          rt_rating_critic      65.583291  25.903849     0.0    47.00    72.0   
          mc_rating_critic      63.131619  16.721461    10.0    52.00    64.0   
Horror    date                2002.731941  19.178227  1900.0  1993.00  2009.0   
          lbxd_rating           59.713664  10.744982    23.6    52.60    60.4   
          imdb_rating           60.084421   9.133391    21.0    54.00    61.0   
          rt_rating_audience    52.791993  20.053658    10.0    37.00    52.0   
          rt_rating_critic      57.251523  27.487411     0.0    35.00    62.0   
          mc_rating_critic      55.735422  17.831030     9.0    43.00    57.0   
Mystery   date                2002.285714  19.760920  1910.0  1996.00  2008.0   
          lbxd_rating           62.631354  10.328827    23.6    55.85    63.4   
          imdb_rating           63.860853   9.014953    32.0    58.00    64.0   
          rt_rating_audience    57.629870  20.946085    10.0    40.00    59.0   
          rt_rating_critic      61.153989  28.308101     0.0    38.00    67.0   
          mc_rating_critic      58.726345  17.612571    14.0    46.00    61.0   
Adventure date                1999.663814  20.959829  1899.0  1991.00  2006.0   
          lbxd_rating           61.381051  11.100700    21.4    54.00    62.6   
          imdb_rating           64.246944  10.010766    17.0    58.00    65.0   
          rt_rating_audience    62.207213  20.036309    10.0    48.00    64.0   
          rt_rating_critic      59.356357  27.284919     0.0    37.00    64.0   
          mc_rating_critic      57.397922  17.390144     1.0    45.00    58.0   
Crime     date                2000.175772  19.092318  1900.0  1991.00  2005.0   
          lbxd_rating           64.106793   9.665935    28.2    57.60    64.6   
          imdb_rating           65.070784   8.517162    25.0    60.00    65.0   
          rt_rating_audience    61.556770  20.683881    10.0    45.00    63.0   
          rt_rating_critic      60.390974  27.943462     0.0    38.00    66.0   
          mc_rating_critic      58.540143  17.858587     7.0    45.00    60.0   

                                 75%     max  
Comedy    date                2014.0  2024.0  
          lbxd_rating           69.4    91.2  
          imdb_rating           69.0    85.0  
          rt_rating_audience    77.0   100.0  
          rt_rating_critic      82.0   100.0  
          mc_rating_critic      68.0    99.0  
Romance   date                2014.0  2024.0  
          lbxd_rating           70.0    88.6  
          imdb_rating           71.0    88.0  
          rt_rating_audience    77.0   100.0  
          rt_rating_critic      84.0

mean        std     min      25%     50%  \
Family    date                1994.372881  23.116816  1899.0  1988.00  2001.0   
          lbxd_rating           60.513559  11.034745    21.4    54.15    62.0   
          imdb_rating           62.036017  11.165840    17.0    55.00    64.0   
          rt_rating_audience    62.735169  19.224006    12.0    48.00    65.0   
          rt_rating_critic      58.514831  28.611457     0.0    33.75    64.0   
          mc_rating_critic      57.218220  18.993156     1.0    43.00    56.0   
Thriller  date                2002.387255  18.626103  1891.0  1994.00  2008.0   
          lbxd_rating           61.894009  10.082886    23.6    55.40    62.6   
          imdb_rating           62.852397   8.906998    25.0    57.00    63.0   
          rt_rating_audience    57.004902  20.733694    10.0    40.00    58.0   
          rt_rating_critic      57.249455  26.914050     0.0    37.00    62.0   
          mc_rating_critic      56.193900  17.527386     9.0    43.00    57.0   
Action    date                2001.958299  18.220491  1891.0  1994.00  2007.0   
          lbxd_rating           60.619075  10.469328    21.4    53.80    61.1   
          imdb_rating           62.632948   9.470637    21.0    57.00    63.0   
          rt_rating_audience    58.741123  20.189608    10.0    43.00    59.0   
          rt_rating_critic      53.129645  27.453308     0.0    30.00    53.0   
          mc_rating_critic      53.844344  17.360078     9.0    41.00    53.0   
History   date                1999.430769  23.166846  1900.0  1993.00  2007.0   
          lbxd_rating           66.854769   7.815090    41.6    62.20    67.0   
          imdb_rating           69.910769   6.363091    50.0    66.00    70.0   
          rt_rating_audience    69.230769  19.385760     0.0    60.00    72.0   
          rt_rating_critic      68.504615  23.439523     8.0    54.00    74.0   
          mc_rating_critic      65.187692  14.963490    22.0    54.00    66.0   
Biography date                2004.798077  17.688397  1912.0  1999.00  2010.5   
          lbxd_rating           67.044231   7.722945    35.4    62.60    67.4   
          imdb_rating           69.795330   6.517198    47.0    66.00    70.0   
          rt_rating_audience    70.436813  18.803300     0.0    61.00    75.0   
          rt_rating_critic      69.560440  21.306222     4.0    57.00    75.0   
          mc_rating_critic      64.715659  14.030334    10.0    55.00    65.0   
Fantasy   date                1998.937666  21.380449  1903.0  1990.00  2006.0   
          lbxd_rating           61.033687  11.189992    24.6    53.60    62.3   
          imdb_rating           62.155172   9.887202    17.0    56.00    63.0   
          rt_rating_audience    59.194960  19.798599    16.0    44.00    59.0   
          rt_rating_critic      56.129973  28.534485     0.0    30.00    62.0   
          mc_rating_critic      55.903183  18.608354     9.0    43.00    56.5   
War       date                1991.718085  25.848805  1915.0  1976.75  2000.0   
          lbxd_rating           68.317021   9.497398    37.6    62.15    68.9   
          imdb_rating           69.202128   7.795624    46.0    64.75    70.0   
          rt_rating_audience    70.361702  17.701235    26.0    56.00    75.0   
          rt_rating_critic      71.021277  24.806623     7.0    56.00    80.0   
          mc_rating_critic      65.819149  16.979990    16.0    54.00    69.5   

                                  75%     max  
Family    date                2009.00  2022.0  
          lbxd_rating           68.00    87.8  
          imdb_rating           70.00    86.0  
          rt_rating_audience    79.00    95.0  
          rt_rating_critic      84.00   100.0  
          mc_rating_critic      74.00    96.0  
Thriller  date                2016.00  2024.0  
          lbxd_rating           68.80    91.2  
          imdb_rating           69.00    86.0  
          rt_rating_audience    75.00   100.0  
          rt_rating_criti

mean        std     min      25%     50%  \
Sci-Fi    date                2001.124346  19.001593  1912.0  1990.00  2008.0   
          lbxd_rating           61.224607  11.178425    22.8    53.75    62.2   
          imdb_rating           62.598168   9.966499    19.0    57.00    63.0   
          rt_rating_audience    56.369110  21.330573    11.0    39.00    58.0   
          rt_rating_critic      56.871728  27.995399     0.0    33.00    60.0   
          mc_rating_critic      56.172775  17.380002     9.0    44.00    57.0   
Western   date                1988.000000  26.575365  1921.0  1966.00  1993.0   
          lbxd_rating           69.242478   8.412340    47.4    63.60    70.0   
          imdb_rating           69.548673   7.363024    55.0    64.00    71.0   
          rt_rating_audience    70.407080  18.390913    27.0    53.00    77.0   
          rt_rating_critic      74.752212  25.059048    11.0    59.00    85.0   
          mc_rating_critic      68.194690  16.529659    25.0    55.00    67.0   
Sport     date                1998.313953  17.629893  1919.0  1988.00  2003.0   
          lbxd_rating           64.170930   8.995325    30.4    58.60    65.5   
          imdb_rating           65.848837   8.782648    21.0    61.00    67.0   
          rt_rating_audience    68.110465  19.480220    14.0    55.50    73.0   
          rt_rating_critic      60.069767  27.784379     0.0    37.00    64.0   
          mc_rating_critic      56.744186  17.319621     7.0    44.00    57.0   
Musical   date                1986.169014  28.260433  1925.0  1962.50  1989.5   
          lbxd_rating           66.500000   9.500750    26.2    62.40    67.9   
          imdb_rating           67.957746   8.971881    19.0    64.00    69.5   
          rt_rating_audience    73.366197  15.494899    14.0    67.25    78.5   
          rt_rating_critic      68.542254  23.832043     7.0    55.00    72.5   
          mc_rating_critic      64.690141  16.950568    14.0    52.00    66.0   
Music     date                2001.144828  17.501747  1927.0  1992.00  2005.0   
          lbxd_rating           65.639310   8.791564    38.6    59.65    66.0   
          imdb_rating           65.427586   9.229668    23.0    61.00    66.0   
          rt_rating_audience    67.634483  17.641887    18.0    57.00    69.5   
          rt_rating_critic      64.237931  24.861456     0.0    48.25    69.0   
          mc_rating_critic      60.644828  16.069315    14.0    50.25    60.5   
Animation date                2003.753012  18.376866  1910.0  1999.75  2009.0   
          lbxd_rating           63.634940  11.643371    24.6    55.80    64.7   
          imdb_rating           67.521084   8.930557    33.0    63.00    68.0   
          rt_rating_audience    67.024096  18.196129    22.0    53.00    69.0   
          rt_rating_critic      66.575301  25.904491     0.0    49.00    73.0   
          mc_rating_critic      61.632530  16.881421    12.0    51.00    61.5   
Film-Noir date                1975.594203  32.191668  1929.0  1947.00  1958.0   
          lbxd_rating           70.179710   8.851796    50.2    65.20    70.0   
          imdb_rating           74.869565   4.039922    68.0    73.00    75.0   
          rt_rating_audience    79.782609  18.028466    27.0    75.00    83.0   
          rt_rating_critic      90.956522  16.723033    37.0    92.00    97.0   
          mc_rating_critic      79.449275  11.489812    61.0    73.00    79.0   

                                  75%     max  
Sci-Fi    date                2015.00  2024.0  
          lbxd_rating           69.20    88.6  
          imdb_rating           68.00    88.0  
          rt_rating_audience    73.00    95.0  
          rt_rating_critic      82.00   100.0  
          mc_rating_critic      68.00    96.0  
Western   date                2014.00  2024.0  
          lbxd_rating           75.40    85.6  
          imdb_rating           76.00    84.0  
          rt_rating_audience    87.00    95.0  
          rt_rating_criti

In [13]:
# Creating bar graphs to compare average rating values for each genre and website
# Creating a dictionary of mean values for each genre based on the genre_summaries dataframe for each website
lbxd_mean_rating_by_genre_dict = {}
imdb_mean_rating_by_genre_dict = {}
rt_aud_mean_rating_by_genre_dict = {}
rt_crit_mean_rating_by_genre_dict = {}
mc_mean_rating_by_genre_dict = {}

for genre in genre_list: # Looping through genre list and appending each mean to dictionary
  lbxd_mean_rating_by_genre_dict[genre] = genre_summaries.loc[(genre,'lbxd_rating'),'mean']
  imdb_mean_rating_by_genre_dict[genre] = genre_summaries.loc[(genre,'imdb_rating'),'mean']
  rt_aud_mean_rating_by_genre_dict[genre] = genre_summaries.loc[(genre,'rt_rating_audience'),'mean']
  rt_crit_mean_rating_by_genre_dict[genre] = genre_summaries.loc[(genre,'rt_rating_critic'),'mean']
  mc_mean_rating_by_genre_dict[genre] = genre_summaries.loc[(genre,'mc_rating_critic'),'mean']

# Sorting each dictionary from lowest to highest mean
lbxd_mean_rating_by_genre_dict = dict(sorted(lbxd_mean_rating_by_genre_dict.items(), key=lambda item: item[1]))
imdb_mean_rating_by_genre_dict = dict(sorted(imdb_mean_rating_by_genre_dict.items(), key=lambda item: item[1]))
rt_aud_mean_rating_by_genre_dict = dict(sorted(rt_aud_mean_rating_by_genre_dict.items(), key=lambda item: item[1]))
rt_crit_mean_rating_by_genre_dict = dict(sorted(rt_crit_mean_rating_by_genre_dict.items(), key=lambda item: item[1]))
mc_mean_rating_by_genre_dict = dict(sorted(mc_mean_rating_by_genre_dict.items(), key=lambda item: item[1]))

# Creating and displaying bar graphs for each dictionary
# All five graphs have been given the same y range (50-95) to more efficiently compare between websites
lbxd_mean_rating_by_genre_graph = px.bar(x=list(lbxd_mean_rating_by_genre_dict.keys()), y=lbxd_mean_rating_by_genre_dict.values(), title="Average values of Letterboxd ratings by genre", labels={'x': 'genre', 'y': 'mean'}, range_y=[50,95])
lbxd_mean_rating_by_genre_graph.update_layout(showlegend=False, width=1200)
lbxd_mean_rating_by_genre_graph.update_traces(marker_color='dodgerblue')
display(lbxd_mean_rating_by_genre_graph)
print("\n")

imdb_mean_rating_by_genre_graph = px.bar(x=list(imdb_mean_rating_by_genre_dict.keys()), y=imdb_mean_rating_by_genre_dict.values(), title="Average values of IMDb ratings by genre", labels={'x': 'genre', 'y': 'mean'}, range_y=[50,95])
imdb_mean_rating_by_genre_graph.update_layout(showlegend=False, width=1200)
imdb_mean_rating_by_genre_graph.update_traces(marker_color='gold')
display(imdb_mean_rating_by_genre_graph)
print("\n")

rt_aud_mean_rating_by_genre_graph = px.bar(x=list(rt_aud_mean_rating_by_genre_dict.keys()), y=rt_aud_mean_rating_by_genre_dict.values(), title="Average values of Rotten Tomatoes audience ratings by genre", labels={'x': 'genre', 'y': 'mean'}, range_y=[50,95])
rt_aud_mean_rating_by_genre_graph.update_layout(showlegend=False, width=1200)
rt_aud_mean_rating_by_genre_graph.update_traces(marker_color='tomato')
display(rt_aud_mean_rating_by_genre_graph)
print("\n")

rt_crit_mean_rating_by_genre_graph = px.bar(x=list(rt_crit_mean_rating_by_genre_dict.keys()), y=rt_crit_mean_rating_by_genre_dict.values(), title="Average values of Rotten Tomatoes critic ratings by genre", labels={'x': 'genre', 'y': 'mean'}, range_y=[50,95])
rt_crit_mean_rating_by_genre_graph.update_layout(showlegend=False, width=1200)
rt_crit_mean_rating_by_genre_graph.update_traces(marker_color='darkred')
display(rt_crit_mean_rating_by_genre_graph)
print("\n")

mc_mean_rating_by_genre_graph = px.bar(x=list(mc_mean_rating_by_genre_dict.keys()), y=mc_mean_rating_by_genre_dict.values(), title="Average values of Metacritic critic ratings by genre", labels={'x': 'genre', 'y': 'mean'}, range_y=[50,95])
mc_mean_rating_by_genre_graph.update_layout(showlegend=False, width=1200)
mc_mean_rating_by_genre_graph.update_traces(marker_color='darkgreen')
display(mc_mean_rating_by_genre_graph)
print("\n")

# Creating a combination of above 5 graphs in order to more accurately compare average ratings by genre
# Doing so by creating a dataframe of mean values by genre rather than a set of dictionaries
genre_means = pd.DataFrame()
for genre in genre_list:
  genre_means = pd.concat([genre_means, genre_summaries.loc[genre,'mean'].rename(genre)], axis=1)
genre_means = genre_means.transpose() # Switching rows and columns
date_means = genre_means.pop('date') # Popping date means column to be used for later
genre_means_graph = px.bar(genre_means,
                           y=['lbxd_rating', 'imdb_rating', 'rt_rating_audience', 'rt_rating_critic', 'mc_rating_critic'],
                           title='Average values for every website by genre', labels={'index': 'genre', 'value': 'mean', 'variable': 'legend'},
                           barmode='group', # barmode='group' is used in order to convert the bar graph into a grouped bar graph
                           range_y=[50,95],
                           color_discrete_map={'lbxd_rating': 'dodgerblue', 'imdb_rating': 'gold', 'rt_rating_audience': 'tomato', 'rt_rating_critic': 'darkred', 'mc_rating_critic': 'darkgreen' # Properly mapping colors accordingly with color_discrete_map
                           })
genre_means_graph.update_layout(xaxis={'categoryorder': 'total ascending'}) # Ordering bars by combined mean values for each genre
display(genre_means_graph)
print("\n")

In [56]:
# Creating a dataframe subset & graph for average release year, using previously popped date means column
date_means = date_means.sort_values()
date_means_graph = px.bar(date_means, title="Average release year by genre", labels={'x': 'genre', 'y': 'mean'}, range_y=[1975,2005])
date_means_graph.update_layout(showlegend=False, width=1200)
date_means_graph.update_traces(marker_color='dimgray')
display(date_means_graph)
print("\n")

# Finding average rating for each year for each website, starting by creating a dataframe for mean values for each year with groupby()
mean_rating_by_year = movie_rating_data.groupby('date').mean(numeric_only=True)
# Adding a column for mean rating between all websites
mean_rating_by_year['all'] = mean_rating_by_year.mean(axis=1)
print("Mean ratings for each website by release year (first & last 10 values)")
display(mean_rating_by_year.head(10))
display(mean_rating_by_year.tail(10))
print("\n")

# Re-displaying unique value curve for date for additional context
display(year_uvcurve)
print("\n")

# Like with genre, all graphs have been given the same range (40-100) for more efficient comparison.
lbxd_mean_rating_by_year_graph = px.bar(mean_rating_by_year['lbxd_rating'], title="Average Letterboxd rating by release year", labels={'value': 'mean'}, range_y=[40,100])
lbxd_mean_rating_by_year_graph.update_layout(showlegend=False)
lbxd_mean_rating_by_year_graph.update_traces(marker_color='dodgerblue')
display(lbxd_mean_rating_by_year_graph)
print("\n")

imdb_mean_rating_by_year_graph = px.bar(mean_rating_by_year['imdb_rating'], title="Average IMDb rating by release year", labels={'value': 'mean'}, range_y=[40,100])
imdb_mean_rating_by_year_graph.update_layout(showlegend=False)
imdb_mean_rating_by_year_graph.update_traces(marker_color='gold')
display(imdb_mean_rating_by_year_graph)
print("\n")

rt_aud_mean_rating_by_year_graph = px.bar(mean_rating_by_year['rt_rating_audience'], title="Average Rotten Tomatoes audience rating by release year", labels={'value': 'mean'}, range_y=[40,100])
rt_aud_mean_rating_by_year_graph.update_layout(showlegend=False)
rt_aud_mean_rating_by_year_graph.update_traces(marker_color='tomato')
display(rt_aud_mean_rating_by_year_graph)
print("\n")

rt_crit_mean_rating_by_year_graph = px.bar(mean_rating_by_year['rt_rating_critic'], title="Average Rotten Tomatoes critic rating by release year", labels={'value': 'mean'}, range_y=[40,100])
rt_crit_mean_rating_by_year_graph.update_layout(showlegend=False)
rt_crit_mean_rating_by_year_graph.update_traces(marker_color='darkred')
display(rt_crit_mean_rating_by_year_graph)
print("\n")

mc_mean_rating_by_year_graph = px.bar(mean_rating_by_year['mc_rating_critic'], title="Average Metacritic critic rating by release year", labels={'value': 'mean'}, range_y=[40,100])
mc_mean_rating_by_year_graph.update_layout(showlegend=False)
mc_mean_rating_by_year_graph.update_traces(marker_color='darkgreen')
display(mc_mean_rating_by_year_graph)
print("\n")

# As much as i would like to make a graph similar to the average values for each website by genre graph, doing the same for date would result in a graph thats way too visually cluttered.
mean_rating_by_year_graph = px.bar(mean_rating_by_year['all'], title="Average movie rating by release year", labels={'value': 'mean'}, range_y=[40,100])
mean_rating_by_year_graph.update_layout(showlegend=False)
mean_rating_by_year_graph.update_traces(marker_color='black')
display(mean_rating_by_year_graph)
print("\n")



Mean ratings for each website by release year (first & last 10 values)


,lbxd_rating,imdb_rating,rt_rating_audience,rt_rating_critic,mc_rating_critic,all
date,,,,,,
1891.0,57.6,66.0,60.0,68.0,68.0,63.92
1895.0,63.0,68.0,64.0,79.0,65.0,67.80
1899.0,69.2,69.0,80.0,97.0,85.0,80.04
1900.0,63.3,71.5,77.0,80.0,60.5,70.46
1903.0,64.5,59.0,59.0,82.0,68.5,66.60
1908.0,63.8,53.0,59.0,67.0,43.0,57.16
1909.0,64.8,64.0,67.0,100.0,67.0,72.56
1910.0,64.6,71.5,86.0,87.0,58.5,73.52
1912.0,61.3,69.0,74.5,73.5,74.0,70.46


,lbxd_rating,imdb_rating,rt_rating_audience,rt_rating_critic,mc_rating_critic,all
date,,,,,,
2015.0,60.115385,62.705128,55.951923,57.112179,55.916667,58.360256
2016.0,60.975691,63.312155,57.502762,61.082873,58.878453,60.350387
2017.0,61.388920,63.911357,60.880886,63.083102,59.900277,61.832909
2018.0,61.468435,64.129973,60.413793,63.472149,60.095491,61.915968
2019.0,61.856986,65.038356,66.142466,66.506849,61.613699,64.231671
2020.0,59.749306,62.888889,61.496528,64.093750,60.114583,61.668611
2021.0,60.031179,63.456274,64.832700,62.996198,60.414449,62.346160
2022.0,60.530952,65.184524,65.297619,62.190476,61.934524,63.027619
2023.0,63.510345,64.632184,62.862069,62.620690,61.505747,63.026207


The following assumptions can be made based on the summary statistics of the entire dataset:
* The unique value curve for date should be taken into consideration when analyzing the data, as a large majority of movies in the dataset are from the 2000s and onward. Additionally, statistics on movies released past 2023 should not be considered reliable, as there is a steep dropoff of the number of movies released past 2022 on the dataset.
* Metacritic critics have the lowest average movie ratings at ~59/100, while IMDb users have the highest average movie ratings, at ~65/100.
* Based on the standard deviation values for the entire dataset, IMDb users have the lowest range of rating values, while Rotten Tomatoes critics have the highest range of rating values. This is further proven by the percentile values being far apart for Rotten Tomatoes critics and closer together for IMDb users.
* Based on the graph of unique values for the dates column, the movie data in my database is skewed to include significantly more data from the 21st century than the 20th century.
* All websites have a similar looking "rating curve" where the most common ratings are near the average rating value, with the exception of the Rotten Tomatoes audience and especially the critic values, which are more sparse in terms of unique rating values. This is further proven by the high standard deviation values for the two columns the rating curves take from.

The following assumptions can be made based on the summary statistics of each genre featured in the dataset:
* The most critically acclaimed genre is Film-Noir, having the highest average rating values for all websites. Critics especially favor Film-Noir based on how high the averages are compared to the second highest average rated genres.
* The above assumption should be taken with a grain of salt, as since Film-Noir is by far the genre with the lowest average release year, its high rating may also be because they favor "older genres". This is further proven by the Western genre, which also has a low average release year and high average rating.
* General audiences seem to not favor Horror, as it has the lowest average rating across all the audience rating values, while critics seem to not favor Action, as it has the lowest average rating across all the critic rating values.

The following assumptions can be made based on the statistics of each website based on movie release years:
* Release year does not have a significant impact on a movie's rating for Letterboxd and IMDb, but for Rotten Tomatoes and Metacritic, there is a dip in average rating for movies from the 2000s to early 2010s, especially for the Rotten Tomatoes critic ratings.
* The average ratings of movies before the 1960s being ever-so-slightly higher than average should be taken with a grain of salt, as a majority of the database includes movies from the 2000s to late 2010s. This is further proven by the significant variations in rating between years (For example, 1929 and 1930).
* The Rotten Tomatoes critic ratings by year graph shows the most sparsity in terms of values.

The assumptions above are only a few examples, more insights can be made with further analysis of the dataset.